In [ ]:
import gc
import torch

# 先清理上一轮 notebook 可能残留的显存缓存。
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print('GPU cache cleared.')
else:
    print('CUDA not available, skip GPU cache clear.')


# 下一课：连续动作空间入门

前面的课程里，我们一直在用像 `CartPole` 这样的离散动作环境：
- 向左
- 向右

但很多更真实的控制任务并不是“二选一”，而是：
- 油门踩多少
- 方向盘打多少
- 机械臂关节转多少角度

这些动作通常是连续值。

这一课的目标不是马上上复杂算法，而是先把最重要的概念打通：
- 什么叫连续动作空间
- 为什么前面的离散动作策略不能直接套过去
- 连续动作策略网络通常怎么输出动作


## 1. 离散动作和连续动作到底差在哪

离散动作环境里，动作集合通常很小：
- `0`
- `1`
- `2`

所以策略网络可以直接输出每个动作的概率：

`[P(a=0), P(a=1), P(a=2)]`

但连续动作环境里，动作不是有限个选项，而是一个区间里的任意实数，比如：

- `a = 0.37`
- `a = -1.42`
- `a = 2.81`

这时候你已经没法再像离散动作那样给每个动作列一个概率表了。


## 2. 连续动作策略通常怎么做

连续动作里，最常见的思路之一是：

- 策略网络不直接输出一个固定动作
- 而是输出一个分布的参数

最经典的例子就是高斯分布（正态分布）：

$a \sim \mathcal{N}(\mu, \sigma)$

也就是说，网络会输出：
- `mean`：动作中心值
- `std`：动作波动大小

然后再从这个分布里采样动作。


In [ ]:
import random
import warnings

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.distributions import Normal

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=3, suppress=True)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


In [ ]:
def pick_device():
    if torch.cuda.is_available():
        try:
            _ = torch.zeros(1, device='cuda')
            return torch.device('cuda')
        except Exception as e:
            warnings.warn(f'检测到 CUDA，但当前环境无法真正使用 GPU，已回退到 CPU。原因: {e}')
    return torch.device('cpu')


device = pick_device()
print('当前设备:', device)
if torch.cuda.is_available():
    print('检测到 CUDA 设备:', torch.cuda.get_device_name(0))


## 3. 先看看一个连续动作环境

我们这节课先不训练，只是先观察一个连续动作环境的接口。

这里选 `Pendulum-v1`，因为它非常经典：
- 目标是让摆杆稳定到竖直方向
- 动作是一个连续扭矩值

这特别适合用来理解连续动作空间是什么样。


In [ ]:
env = gym.make('Pendulum-v1')
state, info = env.reset(seed=42)

print('初始状态:', state)
print('状态维度:', env.observation_space.shape)
print('动作空间:', env.action_space)
print('动作下界:', env.action_space.low)
print('动作上界:', env.action_space.high)


## 4. 为什么不能直接用 Categorical

在离散动作课里，我们常写：

```python
dist = Categorical(logits=logits)
action = dist.sample()
```

这是因为动作只有有限个类别。

但像 `Pendulum` 这种环境，动作是一个连续区间里的实数，
所以这里更自然的分布是：

```python
dist = Normal(mean, std)
action = dist.sample()
```

也就是说：
- 离散动作常用 `Categorical`
- 连续动作常用 `Normal`


## 5. 一个最小连续动作策略网络

这节课的重点是理解结构，所以我们先只做一个最小策略网络：

- 输入：环境状态
- 输出：高斯分布的 `mean`
- `std` 先用一个可学习参数表示

然后我们从这个高斯分布里采样动作。


In [ ]:
def to_tensor(x, device):
    return torch.tensor(x, dtype=torch.float32, device=device)


class ContinuousPolicyNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        self.mean_head = nn.Linear(hidden_dim, action_dim)

        # 用一个可学习的 log_std 来表示动作波动大小。
        self.log_std = nn.Parameter(torch.zeros(action_dim))

    def forward(self, x):
        features = self.net(x)
        mean = self.mean_head(features)
        std = torch.exp(self.log_std).expand_as(mean)
        return mean, std


## 6. 从高斯分布里采样动作

下面这段代码就是连续动作策略最关键的原型：

1. 网络输出 `mean` 和 `std`
2. 构造 `Normal(mean, std)`
3. 采样一个连续动作
4. 计算这个动作的 `log_prob`

这和你之前学 REINFORCE / PPO 时的结构非常像，
只是 `Categorical` 换成了 `Normal`。


In [ ]:
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]

policy_net = ContinuousPolicyNet(state_dim, action_dim, hidden_dim=128).to(device)

state_tensor = to_tensor(state, device).unsqueeze(0)
mean, std = policy_net(state_tensor)
dist = Normal(mean, std)
raw_action = dist.sample()
log_prob = dist.log_prob(raw_action).sum(dim=1)

print('mean:', mean.detach().cpu().numpy())
print('std:', std.detach().cpu().numpy())
print('raw_action:', raw_action.detach().cpu().numpy())
print('log_prob:', log_prob.detach().cpu().numpy())


## 7. 为什么还要做动作裁剪 / squashing

`Pendulum-v1` 的动作范围不是任意实数，而是有上下界的，比如 `[-2, 2]`。

但高斯分布采样出来的值理论上可以无限大或无限小。
所以实际算法里通常会做两件事之一：

- `clip`：直接把动作裁到范围内
- `tanh squashing`：先过 `tanh` 再映射到动作范围

这也是为什么连续动作算法的实现通常会比离散动作更麻烦一点。


In [ ]:
raw_action_np = raw_action.squeeze(0).detach().cpu().numpy()
clipped_action = np.clip(raw_action_np, env.action_space.low, env.action_space.high)

print('raw action:', raw_action_np)
print('clipped action:', clipped_action)


## 8. 实际和环境交互一小段看看

这里我们不正式训练，只是拿当前随机初始化的策略和环境交互几步，
帮助你把“连续动作策略网络”的数据流彻底看清楚。


In [ ]:
demo_env = gym.make('Pendulum-v1')
state, info = demo_env.reset(seed=123)
trajectory = []

with torch.no_grad():
    for step in range(5):
        state_tensor = to_tensor(state, device).unsqueeze(0)
        mean, std = policy_net(state_tensor)
        dist = Normal(mean, std)
        raw_action = dist.sample()
        action = raw_action.squeeze(0).cpu().numpy()
        action = np.clip(action, demo_env.action_space.low, demo_env.action_space.high)

        next_state, reward, terminated, truncated, info = demo_env.step(action)
        trajectory.append((state, action, reward))
        state = next_state
        if terminated or truncated:
            break

for i, (s, a, r) in enumerate(trajectory):
    print(f'step {i}:')
    print('  state =', np.round(s, 3))
    print('  action =', np.round(a, 3))
    print('  reward =', round(float(r), 3))

demo_env.close()


## 9. 这节课最该记住什么

如果你想抓住这节课最核心的变化，就记住：

- 离散动作：输出类别概率，常用 `Categorical`
- 连续动作：输出分布参数，常用 `Normal`

也就是说，连续动作策略不再回答“选哪一类动作”，而是回答：

**“动作大概应该落在哪个连续值附近，波动多大？”**


## 10. 下一课最自然学什么

学完这一课后，最自然的下一步通常是：
- 连续动作版本的 `REINFORCE`
- 连续动作版本的 `PPO`
- 或正式进入 `DDPG / TD3 / SAC`

如果按学习路线来说，我建议下一课先做一个“连续动作版 PPO / Policy Gradient 入门”。
